# **EMIF Group Project 1**
#### Aymen Laraoui, Rijad Talovic, Alexis Gavillet, Clémence Choukroun

## Data Cleaning : Weekly Frequency

Selected Variables
- **WTI** : WTI crude oil futures (CL1 Comdty)
- **HY** : Bloomberg US Corporate High Yield Index (LF98YW Index)
- **SP500** : S&P 500 Index (SPX Index)
- **US2Y** : US Government 2-Year Yield (USGG2YR Index)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Data Loading

In [57]:
raw_path = Path("data/raw/data_hec_projet_1.xlsx")
daily_raw = pd.read_excel(raw_path, sheet_name="Daily", skiprows=5, index_col=0, parse_dates=True)

print("Raw dimensions:", daily_raw.shape)
daily_raw.head()

Raw dimensions: (9443, 13)


,PX_LAST,PX_LAST.1,PX_LAST.2,PX_LAST.3,PX_LAST.4,PX_LAST.5,PX_LAST.6,PX_LAST.7,PX_LAST.8,PX_LAST.9,PX_LAST.10,PX_LAST.11,PX_LAST.12
Dates,,,,,,,,,,,,,
1990-01-01,21.82,19.69,77.8405,NaN,NaN,353.40,567.34,214.69,168.228,7.935,7.841,16.08,401.25
1990-01-02,22.89,19.90,81.0621,NaN,NaN,359.69,568.96,217.30,169.943,7.930,7.875,16.08,399.00
1990-01-03,23.68,20.95,83.0504,NaN,NaN,358.76,569.10,220.45,170.780,7.974,7.927,16.08,395.00
1990-01-04,23.41,20.78,81.1315,NaN,NaN,355.67,571.02,227.30,170.168,7.972,7.910,16.08,396.50
1990-01-05,23.08,20.75,78.7509,NaN,NaN,352.20,566.36,228.91,169.713,7.984,7.885,16.08,405.00


In [73]:
#filtering for selected variables
df = daily_raw.iloc[:, [0, 5, 10, 11]].copy()
df.columns = ["WTI", "SP500", "US2Y", "HY"]

df.head()

,WTI,SP500,US2Y,HY
Dates,,,,
1990-01-01,21.82,353.40,7.841,16.08
1990-01-02,22.89,359.69,7.875,16.08
1990-01-03,23.68,358.76,7.927,16.08
1990-01-04,23.41,355.67,7.910,16.08
1990-01-05,23.08,352.20,7.885,16.08


## 2. Weekly resampling of daily data

In [75]:
# For each week, we retain the last available observation, which approximates the weekly closing value.
df_weekly = df.resample("W-FRI").last()

print(f"Period: {df_weekly.index[0].date()} → {df_weekly.index[-1].date()}")
df_weekly.head()

Period: 1990-01-05 → 2026-03-13


,WTI,SP500,US2Y,HY
Dates,,,,
1990-01-05,23.08,352.20,7.885,16.08
1990-01-12,22.13,339.93,7.938,16.08
1990-01-19,22.08,339.15,8.162,16.08
1990-01-26,22.56,325.80,8.263,16.08
1990-02-02,23.02,330.92,8.341,16.53


## 3. Returns & Excess Returns

In [78]:
df_ret = df_weekly.copy()

#Log-returns
for col in ["WTI", "SP500", "HY"]:
    df_ret[col] = np.log(df_weekly[col]).diff()

df_ret["US2Y"] = df_weekly["US2Y"] /(100*52)

#Excess returns
df_ret["SP500_excess"] = df_ret["SP500"] - df_ret["US2Y"]
df_ret["HY_excess"] = df_ret["HY"] - df_ret["US2Y"]

df_ret.head()

,WTI,SP500,US2Y,HY,SP500_excess,HY_excess
Dates,,,,,,
1990-01-05,NaN,NaN,0.001516,NaN,NaN,NaN
1990-01-12,-0.042032,-0.035459,0.001527,0.000000,-0.036986,-0.001527
1990-01-19,-0.002262,-0.002297,0.001570,0.000000,-0.003867,-0.001570
1990-01-26,0.021506,-0.040159,0.001589,0.000000,-0.041748,-0.001589
1990-02-02,0.020185,0.015593,0.001604,0.027601,0.013989,0.025997


## 4. Final Cleaning & Export

In [87]:
#Remove missing values
df_clean = df_ret.dropna()

df_clean.describe().round(4)

,WTI,SP500,US2Y,HY,SP500_excess,HY_excess
count,1888.0000,1888.0000,1888.0000,1888.0000,1888.0000,1888.0000
mean,0.0007,0.0016,0.0006,-0.0004,0.0009,-0.0011
std,0.0502,0.0232,0.0004,0.0279,0.0232,0.0279
min,-0.3904,-0.2008,0.0000,-0.1619,-0.2012,-0.1619
25%,-0.0243,-0.0102,0.0002,-0.0112,-0.0108,-0.0117
50%,0.0030,0.0028,0.0006,0.0000,0.0023,-0.0012
75%,0.0280,0.0142,0.0010,0.0076,0.0136,0.0071
max,0.2799,0.1142,0.0017,0.2789,0.1142,0.2788


In [91]:
df_clean.to_csv(Path("data/processed/base_nettoyee_hebdo.csv"))
print("Exported : ../data/processed/base_nettoyee_hebdo.csv")

Exported : ../data/processed/base_nettoyee_hebdo.csv


In [8]:
import numpy as np

df_var = df.copy()

# Log-returns pour les variables de prix et de niveau
for col in ["WTI", "SP500"]:
    df_var[col] = np.log(df[col]) - np.log(df[col].shift(1))

# US2Y et HY restent en niveaux (taux / spreads, déjà stationnaires)

# La différence log crée un NaN sur la première ligne
df_var = df_var.dropna()

print("Shape après transformation:", df_var.shape)
print(f"Période : {df_var.index[0].date()} → {df_var.index[-1].date()}")
df_var.head()

Shape après transformation: (9442, 4)
Période : 1990-01-02 → 2026-03-11


,WTI,SP500,US2Y,HY
Date,,,,
1990-01-02,0.047873,0.017642,7.875,16.08
1990-01-03,0.033931,-0.002589,7.927,16.08
1990-01-04,-0.011468,-0.008650,7.910,16.08
1990-01-05,-0.014197,-0.009804,7.885,16.08
1990-01-08,-0.065348,0.004504,7.893,16.08
